<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/typy_wyliczeniowe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_typy_wyliczeniowe.py

from __future__ import annotations

import json
from enum import Enum, IntEnum, Flag, auto, verify, UNIQUE
from pathlib import Path
from typing import Any


try:
    from enum import StrEnum
except ImportError:
    class StrEnum(str, Enum):
        pass


FOLDER_PROJEKTU = Path(
    "/content/drive/MyDrive/projekt_test"
)


@verify(UNIQUE)
class Rynek(StrEnum):
    USA = "usa"
    POLSKA = "polska"
    EUROPA = "europa"
    AZJA = "azja"
    NIEZNANY = "nieznany"


@verify(UNIQUE)
class Sektor(StrEnum):
    TECHNOLOGY = "technology"
    FINANCIALS = "financials"
    HEALTHCARE = "healthcare"
    INDUSTRIALS = "industrials"
    ENERGY = "energy"
    UTILITIES = "utilities"
    MATERIALS = "materials"
    CONSUMER_DISCRETIONARY = "consumer_discretionary"
    CONSUMER_STAPLES = "consumer_staples"
    COMMUNICATION_SERVICES = "communication_services"
    REAL_ESTATE = "real_estate"
    NIEZNANY = "nieznany"


@verify(UNIQUE)
class TypFiltra(StrEnum):
    MOMENTUM = "momentum"
    ZMIENNOSC = "zmiennosc"
    SREDNIA_CENA = "srednia_cena"
    OSTATNIA_CENA = "ostatnia_cena"
    WOLUMEN = "wolumen"


@verify(UNIQUE)
class StatusWyniku(StrEnum):
    PASSED = "passed"
    FAILED = "failed"
    NOT_EVALUATED = "not_evaluated"
    ERROR = "error"


@verify(UNIQUE)
class PoziomRankingu(IntEnum):
    BARDZO_NISKI = 1
    NISKI = 2
    SREDNI = 3
    WYSOKI = 4
    BARDZO_WYSOKI = 5


class WarunekSpolki(Flag):
    BRAK = 0

    MA_DANE_HISTORYCZNE = auto()
    DODATNIE_MOMENTUM = auto()
    NISKA_ZMIENNOSC = auto()
    DODATNI_WOLUMEN = auto()


def parse_rynek(
    wartosc: str | None
) -> Rynek:

    if wartosc is None:
        return Rynek.NIEZNANY

    tekst = str(wartosc).strip().lower()

    mapowanie = {
        "usa": Rynek.USA,
        "us": Rynek.USA,
        "united states": Rynek.USA,
        "united states of america": Rynek.USA,

        "polska": Rynek.POLSKA,
        "poland": Rynek.POLSKA,

        "europa": Rynek.EUROPA,
        "europe": Rynek.EUROPA,

        "azja": Rynek.AZJA,
        "asia": Rynek.AZJA
    }

    return mapowanie.get(
        tekst,
        Rynek.NIEZNANY
    )


def parse_sektor(
    wartosc: str | None
) -> Sektor:

    if wartosc is None:
        return Sektor.NIEZNANY

    tekst = (
        str(wartosc)
        .strip()
        .lower()
        .replace("&", "and")
        .replace(" ", "_")
    )

    mapowanie = {
        "technology":
            Sektor.TECHNOLOGY,

        "information_technology":
            Sektor.TECHNOLOGY,

        "financials":
            Sektor.FINANCIALS,

        "financial":
            Sektor.FINANCIALS,

        "healthcare":
            Sektor.HEALTHCARE,

        "health_care":
            Sektor.HEALTHCARE,

        "industrials":
            Sektor.INDUSTRIALS,

        "industrial":
            Sektor.INDUSTRIALS,

        "energy":
            Sektor.ENERGY,

        "utilities":
            Sektor.UTILITIES,

        "materials":
            Sektor.MATERIALS,

        "consumer_discretionary":
            Sektor.CONSUMER_DISCRETIONARY,

        "consumer_staples":
            Sektor.CONSUMER_STAPLES,

        "communication_services":
            Sektor.COMMUNICATION_SERVICES,

        "real_estate":
            Sektor.REAL_ESTATE
    }

    return mapowanie.get(
        tekst,
        Sektor.NIEZNANY
    )


def parse_typ_filtra(
    wartosc: str
) -> TypFiltra:

    try:
        return TypFiltra(wartosc)

    except ValueError as e:
        raise ValueError(
            f"nieznany typ filtra: {wartosc}"
        ) from e


def parse_status(
    wartosc: str
) -> StatusWyniku:

    try:
        return StatusWyniku(wartosc)

    except ValueError as e:
        raise ValueError(
            f"nieznany status wyniku: {wartosc}"
        ) from e


def wczytaj_model_z_modulu_2(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU
) -> dict[str, Any]:

    ticker = ticker.strip().upper()

    plik = (
        folder
        / f"{ticker}_model.json"
    )

    if not plik.exists():
        raise FileNotFoundError(
            f"brak pliku z modulu 2: {plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8"
    ) as f:

        dane = json.load(f)

    if not isinstance(dane, dict):
        raise ValueError(
            "plik modelu musi zawierac obiekt JSON"
        )

    if "spolka" not in dane:
        raise ValueError(
            "brak sekcji spolka w danych z modulu 2"
        )

    if "statystyki" not in dane:
        raise ValueError(
            "brak sekcji statystyki w danych z modulu 2"
        )

    if "notowania" not in dane:
        raise ValueError(
            "brak sekcji notowania w danych z modulu 2"
        )

    return dane


def okresl_warunki(
    dane: dict[str, Any]
) -> WarunekSpolki:

    warunki = WarunekSpolki.BRAK

    notowania = dane.get(
        "notowania",
        []
    )

    statystyki = dane.get(
        "statystyki",
        {}
    )

    if notowania:
        warunki |= (
            WarunekSpolki.MA_DANE_HISTORYCZNE
        )

    momentum = statystyki.get(
        TypFiltra.MOMENTUM.value
    )

    if (
        momentum is not None
        and float(momentum) > 0
    ):
        warunki |= (
            WarunekSpolki.DODATNIE_MOMENTUM
        )

    zmiennosc = statystyki.get(
        TypFiltra.ZMIENNOSC.value
    )

    if (
        zmiennosc is not None
        and float(zmiennosc) < 2.0
    ):
        warunki |= (
            WarunekSpolki.NISKA_ZMIENNOSC
        )

    if notowania:
        ostatnie = notowania[-1]

        volume = ostatnie.get(
            "volume",
            0
        )

        if float(volume) > 0:
            warunki |= (
                WarunekSpolki.DODATNI_WOLUMEN
            )

    return warunki


def warunki_do_listy(
    warunki: WarunekSpolki
) -> list[str]:

    wynik: list[str] = []

    for warunek in WarunekSpolki:

        if (
            warunek != WarunekSpolki.BRAK
            and warunek in warunki
        ):
            wynik.append(
                warunek.name
            )

    return wynik


def przygotuj_dane_enum(
    dane: dict[str, Any]
) -> dict[str, Any]:

    spolka = dane["spolka"]

    kraj = spolka.get(
        "kraj"
    )

    sektor_raw = spolka.get(
        "sektor"
    )

    rynek = parse_rynek(
        kraj
    )

    sektor = parse_sektor(
        sektor_raw
    )

    status = (
        StatusWyniku.NOT_EVALUATED
    )

    ranking = (
        PoziomRankingu.SREDNI
    )

    warunki = okresl_warunki(
        dane
    )

    wynik = {
        "spolka": {
            **spolka,

            "rynek":
                rynek.value,

            "sektor":
                sektor.value
        },

        "typy_filtrow": [
            TypFiltra.MOMENTUM.value,
            TypFiltra.ZMIENNOSC.value,
            TypFiltra.SREDNIA_CENA.value,
            TypFiltra.OSTATNIA_CENA.value,
            TypFiltra.WOLUMEN.value
        ],

        "status":
            status.value,

        "ranking":
            ranking.value,

        "warunki": {
            "wartosc_flag":
                warunki.value,

            "spelnione":
                warunki_do_listy(
                    warunki
                )
        },

        "statystyki":
            dane["statystyki"],

        "notowania":
            dane["notowania"]
    }

    return wynik


def zapisz_dane_enum(
    ticker: str,
    dane: dict[str, Any],
    folder: Path = FOLDER_PROJEKTU
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    ticker = ticker.strip().upper()

    plik = (
        folder
        / f"{ticker}_enum.json"
    )

    with open(
        plik,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2
        )

    return plik


def pokaz_enumy() -> None:

    print("\nRYNEK")

    for element in Rynek:
        print(
            element.name,
            "->",
            element.value
        )

    print("\nSEKTOR")

    for element in Sektor:
        print(
            element.name,
            "->",
            element.value
        )

    print("\nTYP FILTRA")

    for element in TypFiltra:
        print(
            element.name,
            "->",
            element.value
        )

    print("\nSTATUS WYNIKU")

    for element in StatusWyniku:
        print(
            element.name,
            "->",
            element.value
        )

    print("\nPOZIOM RANKINGU")

    for element in PoziomRankingu:
        print(
            element.name,
            "->",
            element.value
        )

    print("\nFLAG")

    for element in WarunekSpolki:
        print(
            element.name,
            "->",
            element.value
        )


def run() -> None:

    ticker = input(
        "podaj ticker: "
    ).strip().upper()

    print(
        "\nwczytywanie danych "
        "zapisanych przez modul 2..."
    )

    dane = (
        wczytaj_model_z_modulu_2(
            ticker
        )
    )

    print(
        "wczytano model:",
        ticker
    )

    wynik = przygotuj_dane_enum(
        dane
    )

    plik = zapisz_dane_enum(
        ticker,
        wynik
    )

    print(
        "\nRYNEK:",
        wynik["spolka"]["rynek"]
    )

    print(
        "SEKTOR:",
        wynik["spolka"]["sektor"]
    )

    print(
        "STATUS:",
        wynik["status"]
    )

    print(
        "RANKING:",
        wynik["ranking"]
    )

    print(
        "FLAG VALUE:",
        wynik["warunki"]["wartosc_flag"]
    )

    print(
        "SPELNIONE WARUNKI:"
    )

    for warunek in wynik[
        "warunki"
    ]["spelnione"]:

        print(
            "-",
            warunek
        )

    print(
        "\nzapisano dane "
        "dla kolejnego modulu:"
    )

    print(
        plik
    )

    pokaz_enumy()

    print(
        "\nMODUL TYPOW WYLICZENIOWYCH "
        "DZIALA POPRAWNIE"
    )